In [77]:
using JuMP
try import SCS;   catch err; println("SCS not installed"); end

# Lab 3

## Example of an infeasibile problem:
###  $\max (x_1 + x_2)$ such that   $x_1 \leq 0$, $x_1 \geq 2$, $x_1 + x_2 = 3$

In [86]:
#implementation
model = Model(SCS.Optimizer)
@variable(model, x[1:2])
A = [ 1 0; 
     -1 0;
      2 2]
b = [0; -2; 3]
c = [1; 1];

In [87]:
@constraint(model, A*x ≤ b)
@objective(model, Max, c'*x);
print(model)

In [88]:
optimize!(model)
solution_summary(model)

------------------------------------------------------------------
	       SCS v3.2.8 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 2, constraints m: 3
cones: 	  l: linear vars: 3
settings: eps_abs: 1.0e-04, eps_rel: 1.0e-04, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
	  compiled with openmp parallelization enabled
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 4, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  scale  | time (s)
------------------------------------------------------------------
     0| 3.13e+01  1.00e+00  1.49e+01 -9.67e+00  1.00e-01  4.61e-03 
    25| 1.21e+14  2.30e+10  1.28e+19  7.24e+18  1.00e-01  1.16e-02 
--------------------------------

solution_summary(; result = 1, verbose = false)
├ solver_name          : SCS
├ Termination
│ ├ termination_status : INFEASIBLE
│ ├ result_count       : 1
│ └ raw_status         : infeasible
├ Solution (result = 1)
│ ├ primal_status        : INFEASIBLE_POINT
│ ├ dual_status          : INFEASIBILITY_CERTIFICATE
│ ├ objective_value      : NaN
│ └ dual_objective_value : -1.00000e+00
└ Work counters
  └ solve_time (sec)   : 1.16115e-02

## we see: termination status is infeasible. By Farkas Lemma, this means the dual must be unbounded!
## lets find a certificate. Write down the dual with c=0 and constraint b^T y = -1

In [89]:
#implementation
model = Model(SCS.Optimizer)
@variable(model, y[1:3] ≥ 0)
@constraint(model, b'*y == -1)
@constraint(model, A'*y == [0; 0])
@objective(model, Min, 0);
print(model)

In [91]:
optimize!(model)
solution_summary(model)

------------------------------------------------------------------
	       SCS v3.2.8 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 3, constraints m: 6
cones: 	  z: primal zero / dual free vars: 3
	  l: linear vars: 3
settings: eps_abs: 1.0e-04, eps_rel: 1.0e-04, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
	  compiled with openmp parallelization enabled
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 9, nnz(P): 0
------------------------------------------------------------------
 iter | pri res | dua res |   gap   |   obj   |  scale  | time (s)
------------------------------------------------------------------
     0| 9.95e-01  1.51e-01  1.26e-01  6.28e-02  1.00e-01  5.13e-03 
    25| 9.06e-09  3.45e-08  3.27e-08  1.64e-08  1.00e-01  1.11

solution_summary(; result = 1, verbose = false)
├ solver_name          : SCS
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ └ raw_status         : solved
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 0.00000e+00
│ └ dual_objective_value : 3.27105e-08
└ Work counters
  └ solve_time (sec)   : 1.11207e-02

## The solution summary tells us that the dual is feasible. This implies that the (original) primal is infeasible.
## Note that the solver does not know which problem you label as dual or primal. 
## The solver calls any problem you give it a "primal problem".

### By Farkas Lemma, the below infeasibility certificate proves that $Ax \leq b$   has no solution.

In [93]:
value.(y)

3-element Vector{Float64}:
 0.4999999964592494
 0.5000000030903352
 4.531587385512516e-9

## Also note that the modified dual problem has it's own corresponding primal, which by the duality theorem is feasible!